# Module 01: Probability Foundations

## Intermediate Statistics for Econometrics

In this module, we establish the mathematical foundations of probability needed for statistical inference in econometrics. We'll explore both frequentist and Bayesian perspectives, learn the axioms that govern probability, and develop intuition for conditional probability, independence, and causal structures.

By the end of this module, you will:
- Understand the philosophical differences between frequentist and Bayesian probability
- Master conditional probability and Bayes' theorem
- Identify when independence assumptions hold (and when they fail)
- Recognize confounders and mediators in causal reasoning
- Work with continuous random variables and probability densities

## Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

# Configure plotting
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams['figure.figsize'] = (10, 6)


---

## 1. Frequentist vs. Bayesian Probability

### The Frequentist Perspective
**Probability** is the long-run relative frequency of an event in repeated experiments under identical conditions.

For example: "If we could rerun the Italian economy millions of times under identical conditions, Italian GDP would grow > 2% in X% of those runs."

### The Bayesian Perspective
**Probability** represents our degree of belief or uncertainty about an event, conditioned on available information.

For example: "Given current economic indicators, inflation data, and policy settings, I assign a 65% probability that Italian GDP will grow > 2% this year."

### Key Practical Difference
- **Frequentist**: Parameters are fixed (unknown); data are random.
- **Bayesian**: Parameters are random variables; we update our beliefs as data arrive.

### Italian GDP Example
Suppose we want to assess P(Italian GDP growth > 2%). 

In [ ]:
# Frequentist approach: Look at historical data
years_above_2pct = 18  # number of years in last 50 where growth > 2%
total_years = 50
frequentist_prob = years_above_2pct / total_years

print(f"Frequentist estimate: P(GDP growth > 2%) = {frequentist_prob:.2%}")
print(f"Based on {years_above_2pct} out of {total_years} historical years.")

# Bayesian approach: Combine historical data with current information
prior_belief = 0.50  # Before seeing 2024 data, we're uncertain
likelihood_improvement = 1.2  # Recent economic data suggests better conditions
bayesian_posterior = prior_belief * likelihood_improvement / (
    prior_belief * likelihood_improvement + (1 - prior_belief)
)

print(f"\nBayesian estimate: P(GDP growth > 2%) = {bayesian_posterior:.2%}")
print(f"Combines historical rate ({frequentist_prob:.2%}) with current conditions.")

---

## 2. Axioms of Probability (Kolmogorov)

All of probability theory rests on three foundational axioms. These rules must hold for any valid probability function P(·):

### Axiom 1: Non-negativity
$$P(A) \geq 0 \quad \text{for all events } A$$

Probabilities cannot be negative.

### Axiom 2: Normalization
$$P(\Omega) = 1$$

where $\Omega$ is the sample space (the set of all possible outcomes). The total probability across all outcomes is 1.

### Axiom 3: Countable Additivity
For mutually exclusive events $A_1, A_2, A_3, \ldots$ (i.e., no overlap):
$$P\left(\bigcup_{i=1}^{\infty} A_i\right) = \sum_{i=1}^{\infty} P(A_i)$$

The probability of a union of disjoint events equals the sum of their individual probabilities.

### Simple Examples

In [ ]:
# Example 1: Fair die
print("=== Fair Die ===")
outcomes = list(range(1, 7))
p_each = 1/6

# Axiom 1: Non-negativity ✓
print(f"Axiom 1 (Non-negativity): Each outcome has P(X) = {p_each:.4f} >= 0 ✓")

# Axiom 2: Normalization ✓
total_prob = sum([p_each for _ in outcomes])
print(f"Axiom 2 (Normalization): Sum of all probabilities = {total_prob} ✓")

# Axiom 3: Additivity
# P(X ∈ {1,2,3}) = P(X=1) + P(X=2) + P(X=3)
p_1_to_3 = 3 * p_each
print(f"Axiom 3 (Additivity): P(X ∈ {{1,2,3}}) = 3 × {p_each:.4f} = {p_1_to_3:.4f} ✓")

print("\n=== Italian Worker Education ===")
# P(Degree) + P(No Degree) = 1
p_degree = 0.30
p_no_degree = 0.70
print(f"P(Degree) = {p_degree:.2f}")
print(f"P(No Degree) = {p_no_degree:.2f}")
print(f"Sum = {p_degree + p_no_degree:.2f} (Axiom 2: must equal 1.0) ✓")

---

## 3. Conditional Probability

### Definition
The **conditional probability** of event A given that event B has occurred is:

$$P(A|B) = \frac{P(A \cap B)}{P(B)}$$

provided that $P(B) > 0$.

**Intuition**: Of all the outcomes where B happens, what fraction also have A?

### Worked Example: Italian Workers

Consider a large population of Italian workers. We know:
- 30% have a university degree
- 20% earn more than €50,000 per year
- 15% have both a degree AND earn > €50,000

**Question**: Given that a randomly selected worker has a degree, what's the probability they earn > €50,000?

In [ ]:
# Define events
p_degree = 0.30
p_high_income = 0.20
p_degree_and_income = 0.15

# Compute P(Income > 50k | Degree)
p_income_given_degree = p_degree_and_income / p_degree

print("=== Conditional Probability Example ===")
print(f"P(Degree) = {p_degree}")
print(f"P(Income > €50k) = {p_high_income}")
print(f"P(Degree AND Income > €50k) = {p_degree_and_income}")
print()
print(f"P(Income > €50k | Degree) = P(Degree AND Income) / P(Degree)")
print(f"                          = {p_degree_and_income} / {p_degree}")
print(f"                          = {p_income_given_degree:.2f}")
print()
print(f"Interpretation: Among workers with a degree, {p_income_given_degree:.0%} earn > €50k.")

### Visual Representation

In [ ]:
# Create visual representation
# Population counts: 1500 with both, 1000 degree only, 500 income only, 8000 neither
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Full population breakdown
categories = ['Both\n(D & I)', 'Degree\nonly', 'Income\nonly', 'Neither']
counts = [1500, 1000, 500, 7000]
colors = ['#2ecc71', '#3498db', '#e74c3c', '#95a5a6']

ax = axes[0]
bars = ax.bar(categories, counts, color=colors, edgecolor='black', linewidth=2)
ax.set_ylabel('Number of Workers', fontsize=12)
ax.set_title('Population Breakdown (n=10,000)', fontsize=13, fontweight='bold')
ax.set_ylim(0, 8000)

# Add percentage labels on bars
for bar, count in zip(bars, counts):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{count}\n({count/10000:.0%})',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

# Right: Conditional probability focus
ax = axes[1]
degree_categories = ['High Income\n(€50k+)', 'Moderate Income\n(<€50k)']
degree_counts = [1500, 1000]
colors_cond = ['#2ecc71', '#ecf0f1']

bars = ax.bar(degree_categories, degree_counts, color=colors_cond, edgecolor='black', linewidth=2)
ax.set_ylabel('Number of Workers', fontsize=12)
ax.set_title('Among Workers WITH Degree (n=2,500)', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1800)

for bar, count in zip(bars, degree_counts):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{count}\n({count/2500:.0%})',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('conditional_probability.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"P(Income > €50k | Degree) = 1500 / 2500 = {1500/2500:.2f}")


---

## 4. Bayes' Theorem

### Derivation from Conditional Probability

Starting from the definition of conditional probability:
$$P(A|B) = \frac{P(A \cap B)}{P(B)}$$

We can rearrange to get:
$$P(A \cap B) = P(A|B) \cdot P(B)$$

By symmetry:
$$P(A \cap B) = P(B|A) \cdot P(A)$$

Setting these equal:
$$P(B|A) \cdot P(A) = P(A|B) \cdot P(B)$$

Solving for P(A|B):

$$\boxed{P(A|B) = \frac{P(B|A) \cdot P(A)}{P(B)}}$$

### Terminology
- **P(A|B)**: Posterior probability (what we want to know)
- **P(B|A)**: Likelihood (how likely the data are if A is true)
- **P(A)**: Prior probability (our belief before seeing data B)
- **P(B)**: Evidence or marginal likelihood (the total probability of observing B)

### Worked Example: Disease Testing in the Italian Healthcare System

A rare disease affects 0.1% of the population. A diagnostic test is 99% accurate (correctly identifies both sick and healthy people). If a person tests positive, what's the probability they actually have the disease?

In [ ]:
# Define the probabilities
p_disease = 0.001  # Prior: 0.1% have the disease
p_test_pos_given_disease = 0.99  # Likelihood: P(Test+ | Disease)
p_test_pos_given_no_disease = 0.01  # False positive rate
p_no_disease = 1 - p_disease

# Calculate P(Test+) using the law of total probability
p_test_pos = (p_test_pos_given_disease * p_disease + 
               p_test_pos_given_no_disease * p_no_disease)

# Apply Bayes' theorem
p_disease_given_test_pos = (p_test_pos_given_disease * p_disease) / p_test_pos

print("=== Bayes' Theorem: Disease Testing Example ===")
print(f"Prior: P(Disease) = {p_disease:.4f} (0.1%)")
print(f"Likelihood: P(Test+ | Disease) = {p_test_pos_given_disease:.2f}")
print(f"False positive rate: P(Test+ | No disease) = {p_test_pos_given_no_disease:.2f}")
print(f"\nEvidence: P(Test+) = {p_test_pos:.4f}")
print()
print(f"Posterior: P(Disease | Test+) = {p_disease_given_test_pos:.4f}")
print(f"\nInterpretation: Even with a positive test, there's only a {p_disease_given_test_pos:.1%}")
print(f"chance the person actually has the disease!")
print()
print(f"Why? Because the disease is very rare (prior = 0.1%), so false positives")
print(f"outnumber true positives in the population.")

### Visualizing Bayes' Theorem

In [ ]:
# Simulate population
population = 1_000_000

# Calculate actual numbers
n_disease = int(population * p_disease)
n_no_disease = population - n_disease

n_test_pos_disease = int(n_disease * p_test_pos_given_disease)
n_test_pos_no_disease = int(n_no_disease * p_test_pos_given_no_disease)
n_test_pos_total = n_test_pos_disease + n_test_pos_no_disease

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Prior - population composition
ax = axes[0]
prior_labels = ['Disease', 'No Disease']
prior_counts = [n_disease, n_no_disease]
colors = ['#e74c3c', '#2ecc71']

bars = ax.bar(prior_labels, prior_counts, color=colors, edgecolor='black', linewidth=2)
ax.set_ylabel('Number of People', fontsize=12)
ax.set_title('Prior: Population Before Testing', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1_000_000)

for bar, count in zip(bars, prior_counts):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{count:,}\n({count/population:.2%})',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

# Right: Posterior - among those who test positive
ax = axes[1]
posterior_labels = ['True Disease', 'False Positive']
posterior_counts = [n_test_pos_disease, n_test_pos_no_disease]
colors_post = ['#e74c3c', '#f39c12']

bars = ax.bar(posterior_labels, posterior_counts, color=colors_post, edgecolor='black', linewidth=2)
ax.set_ylabel('Number of People', fontsize=12)
ax.set_title(f'Posterior: Among Test+ Results (n={n_test_pos_total:,})', fontsize=13, fontweight='bold')
ax.set_ylim(0, max(posterior_counts) * 1.2)

for bar, count in zip(bars, posterior_counts):
    height = bar.get_height()
    pct = count / n_test_pos_total
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{count:,}\n({pct:.2%})',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('bayes_theorem.png', dpi=100, bbox_inches='tight')
plt.show()

---

## 5. Independence

### Definition
Two events A and B are **independent** if:

$$P(A \cap B) = P(A) \cdot P(B)$$

Equivalently:
$$P(A|B) = P(A)$$

**Intuition**: Knowing that B occurred tells us nothing about the probability of A.

### Checking Independence in the Workers Example

In [ ]:
# Recall the Italian workers example
p_degree = 0.30
p_high_income = 0.20
p_degree_and_income = 0.15

# Under independence, we would expect:
p_under_independence = p_degree * p_high_income

print("=== Testing Independence ===")
print(f"P(Degree) × P(Income > €50k) = {p_degree} × {p_high_income} = {p_under_independence:.2f}")
print(f"But actual P(Degree AND Income > €50k) = {p_degree_and_income:.2f}")
print()

if abs(p_degree_and_income - p_under_independence) < 0.001:
    print("✓ Events are independent.")
else:
    print("✗ Events are NOT independent.")
    difference = p_degree_and_income - p_under_independence
    print(f"\nDifference: {p_degree_and_income:.2f} - {p_under_independence:.2f} = {difference:.2f}")
    print(f"\nInterpretation: Having a degree increases the probability of earning > €50k.")
    print(f"Events are positively correlated.")

### Why Independence Matters in Econometrics: Omitted Variable Bias

Independence is crucial in econometrics. Consider a simple regression:
$$\text{Income} = \beta_0 + \beta_1 \cdot \text{Education} + u$$

where $u$ is the error term.

**OLS assumes** that the error $u$ is independent of the regressor (Education). If this fails—for example, if both Education and Income depend on a third variable like "ability" or "family background"—then the estimate $\hat{\beta}_1$ will be **biased**.

**Example of the problem**:
- People from wealthy families are more likely to get degrees (Education ↑)
- People from wealthy families earn more even without a degree (Error $u$ ↑)
- Therefore, Education and $u$ are not independent
- The estimated return to education is inflated (we attribute family wealth effects to education)

This is the essence of **omitted variable bias**: failure of independence due to missing confounders.

---

## 6. Confounders vs. Mediators

### Definitions Using a Causal DAG

Consider the relationship between three variables:
- **Education** (E): Years of schooling
- **Sector** (S): Industry of employment (Manufacturing, Tech, Services)
- **Income** (I): Annual earnings

### Scenario 1: Sector as a Confounder

```
    Sector
    /    \
   E  →   I
```

**Causal structure**:
- Sector influences both Education and Income
- Tech sector: hires educated workers and pays well
- Manufacturing: lower education requirements, lower pay

If we estimate the effect of Education on Income **without controlling for Sector**, we'll overestimate it. The observed correlation between E and I is partly due to their common cause (Sector).

**What to do**: Include Sector in the regression to isolate E → I.

### Scenario 2: Sector as a Mediator

```
E → Sector → I
```

**Causal structure**:
- Education influences choice of Sector
- Sector influences Income
- Example: More educated workers enter tech (higher pay); less educated enter manufacturing (lower pay)

If we're interested in the **total effect** of Education on Income, we should NOT control for Sector. Controlling for Sector removes part of the causal path and estimates only the "direct" effect.

If we're interested in the **direct effect** (E → I not through Sector), we SHOULD control for Sector.

### Visual Comparison

In [ ]:
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Confounder
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Sector as CONFOUNDER', fontsize=13, fontweight='bold', pad=20)

# Nodes
circle_sector = plt.Circle((5, 8), 0.5, color='#3498db', ec='black', linewidth=2)
circle_edu = plt.Circle((2, 3), 0.5, color='#2ecc71', ec='black', linewidth=2)
circle_income = plt.Circle((8, 3), 0.5, color='#e74c3c', ec='black', linewidth=2)

ax.add_patch(circle_sector)
ax.add_patch(circle_edu)
ax.add_patch(circle_income)

ax.text(5, 8, 'Sector', ha='center', va='center', fontsize=11, fontweight='bold', color='white')
ax.text(2, 3, 'Education', ha='center', va='center', fontsize=11, fontweight='bold', color='white')
ax.text(8, 3, 'Income', ha='center', va='center', fontsize=11, fontweight='bold', color='white')

# Arrows
arrow1 = FancyArrowPatch((4.3, 7.5), (2.5, 3.5), arrowstyle='->', mutation_scale=30, 
                          linewidth=2.5, color='#3498db')
arrow2 = FancyArrowPatch((5.7, 7.5), (7.5, 3.5), arrowstyle='->', mutation_scale=30,
                          linewidth=2.5, color='#3498db')
arrow3 = FancyArrowPatch((2.5, 2.5), (7.5, 2.5), arrowstyle='->', mutation_scale=30,
                          linewidth=2.5, color='#2ecc71')

ax.add_patch(arrow1)
ax.add_patch(arrow2)
ax.add_patch(arrow3)

ax.text(3.2, 5.5, 'Causes', fontsize=9, style='italic', color='#3498db')
ax.text(6.8, 5.5, 'Causes', fontsize=9, style='italic', color='#3498db')
ax.text(5, 2, 'Direct effect', fontsize=9, style='italic', color='#2ecc71')

ax.text(5, 0.5, 'Problem: Observed E→I correlation includes\nthe effect of their common cause (Sector).\n\nSolution: Control for Sector to isolate E→I.',
        ha='center', fontsize=10, bbox=dict(boxstyle='round', facecolor='#ecf0f1', alpha=0.8))

# Right: Mediator
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Sector as MEDIATOR', fontsize=13, fontweight='bold', pad=20)

# Nodes
circle_edu = plt.Circle((2, 5), 0.5, color='#2ecc71', ec='black', linewidth=2)
circle_sector = plt.Circle((5, 5), 0.5, color='#3498db', ec='black', linewidth=2)
circle_income = plt.Circle((8, 5), 0.5, color='#e74c3c', ec='black', linewidth=2)

ax.add_patch(circle_edu)
ax.add_patch(circle_sector)
ax.add_patch(circle_income)

ax.text(2, 5, 'Education', ha='center', va='center', fontsize=11, fontweight='bold', color='white')
ax.text(5, 5, 'Sector', ha='center', va='center', fontsize=11, fontweight='bold', color='white')
ax.text(8, 5, 'Income', ha='center', va='center', fontsize=11, fontweight='bold', color='white')

# Arrows
arrow1 = FancyArrowPatch((2.5, 5), (4.5, 5), arrowstyle='->', mutation_scale=30,
                          linewidth=2.5, color='#2ecc71')
arrow2 = FancyArrowPatch((5.5, 5), (7.5, 5), arrowstyle='->', mutation_scale=30,
                          linewidth=2.5, color='#3498db')

ax.add_patch(arrow1)
ax.add_patch(arrow2)

ax.text(3.5, 5.4, 'Causes', fontsize=9, style='italic', color='#2ecc71')
ax.text(6.5, 5.4, 'Causes', fontsize=9, style='italic', color='#3498db')

ax.text(5, 1, 'Situation: E influences Income partly through Sector.\nSector is on the causal path (mediator).\n\nDon\'t control: Total effect = direct + indirect.\nDo control: Direct effect only.',
        ha='center', fontsize=10, bbox=dict(boxstyle='round', facecolor='#ecf0f1', alpha=0.8))

plt.tight_layout()
plt.savefig('confounder_mediator.png', dpi=100, bbox_inches='tight')
plt.show()

print("Confounder vs. Mediator: Key Distinction")
print("="*50)
print("\nCONFOUNDER (common cause):")
print("  → Has arrows pointing TO both E and I")
print("  → Creates spurious correlation")
print("  → MUST control to get causal effect of E on I")
print("\nMEDIATOR (on causal path):")
print("  → Sits between E and I")
print("  → Part of the mechanism E → I")
print("  → Don't control if you want TOTAL effect")
print("  → Do control if you want DIRECT effect (excluding mediator path)")

---

## 7. Continuous Random Variables

### Probability Density Function (PDF)

For a continuous random variable $X$, the **probability density function** $f(x)$ describes the relative likelihood of values. Key properties:

1. **Non-negativity**: $f(x) \geq 0$ for all $x$
2. **Total area under curve = 1**: $\int_{-\infty}^{\infty} f(x) \, dx = 1$
3. **Probability of a range**: $P(a \leq X \leq b) = \int_a^b f(x) \, dx$

### Cumulative Distribution Function (CDF)

The **cumulative distribution function** gives the probability of observing a value less than or equal to $x$:

$$F(x) = P(X \leq x) = \int_{-\infty}^{x} f(t) \, dt$$

The probability of a range can be computed from the CDF:
$$P(a \leq X \leq b) = F(b) - F(a)$$

### Critical Insight: Point Probabilities are Zero

For continuous distributions:
$$P(X = x) = 0$$

**Why?** Because $x$ is just a single point on the real line, which has infinitesimal width. Probability is concentrated on intervals, not points.

### Important: Density Can Exceed 1

The **density** $f(x)$ can be greater than 1! What matters is that the **total area under the curve** integrates to 1. 

**Example**: Uniform distribution on $[0, 0.1]$
- For the distribution to integrate to 1 over an interval of width 0.1, the height must be 1/0.1 = 10
- So $f(x) = 10$ for $x \in [0, 0.1]$

In [ ]:
# Example: Uniform distribution on [0, 0.1]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PDF
ax = axes[0]
x_uniform = np.linspace(-0.05, 0.15, 1000)
# Uniform density on [0, 0.1] has height 10
y_uniform = np.where((x_uniform >= 0) & (x_uniform <= 0.1), 10, 0)

ax.fill_between(x_uniform, y_uniform, alpha=0.3, color='#3498db', label='f(x)')
ax.plot(x_uniform, y_uniform, 'b-', linewidth=2.5)
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('Density f(x)', fontsize=12)
ax.set_title('PDF: Uniform on [0, 0.1]', fontsize=13, fontweight='bold')
ax.set_ylim(0, 12)
ax.grid(True, alpha=0.3)
ax.axhline(y=1, color='red', linestyle='--', linewidth=1.5, label='y = 1')
ax.legend()

# Annotation
ax.text(0.05, 5, f'Height = 1/0.1 = 10\nArea = 10 × 0.1 = 1 ✓', 
        ha='center', fontsize=11, bbox=dict(boxstyle='round', facecolor='#ecf0f1', alpha=0.9))

# CDF
ax = axes[1]
x_cdf = np.linspace(-0.05, 0.15, 1000)
y_cdf = np.where(x_cdf < 0, 0, np.where(x_cdf <= 0.1, 10 * x_cdf, 1))

ax.plot(x_cdf, y_cdf, 'g-', linewidth=2.5)
ax.fill_between(x_cdf, y_cdf, alpha=0.3, color='#2ecc71')
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('Probability F(x)', fontsize=12)
ax.set_title('CDF: Uniform on [0, 0.1]', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3)

# Show P(0.03 <= X <= 0.07)
x_range = [0.03, 0.07]
prob_range = 10 * (0.07 - 0.03)
ax.axvline(x=0.03, color='red', linestyle=':', alpha=0.5)
ax.axvline(x=0.07, color='red', linestyle=':', alpha=0.5)
ax.plot([0.03, 0.07], [0.3, 0.7], 'r-', linewidth=2, label=f'P(0.03 ≤ X ≤ 0.07) = 0.4')
ax.legend()

plt.tight_layout()
plt.savefig('continuous_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

print("Uniform Distribution on [0, 0.1]")
print("="*50)
print(f"PDF: f(x) = 10 for x ∈ [0, 0.1]")
print(f"     f(x) = 0 otherwise")
print(f"\nTotal area: 10 × 0.1 = {10 * 0.1} ✓")
print(f"\nNote: f(x) = 10 > 1, which is fine for continuous distributions!")
print(f"\nExamples:")
print(f"  P(X ≤ 0.05) = F(0.05) = 10 × 0.05 = 0.50")
print(f"  P(0.03 ≤ X ≤ 0.07) = F(0.07) - F(0.03) = 0.70 - 0.30 = 0.40")
print(f"  P(X = 0.05) = 0 (point has zero probability)")

### Normal Distribution Example

In [ ]:
# Normal distribution
mu, sigma = 100, 15  # IQ scores: mean 100, SD 15

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PDF
ax = axes[0]
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 1000)
y = stats.norm.pdf(x, mu, sigma)

ax.fill_between(x, y, alpha=0.3, color='#9b59b6')
ax.plot(x, y, 'purple', linewidth=2.5)
ax.axvline(x=mu, color='red', linestyle='--', linewidth=2, label=f'μ = {mu}')
ax.set_xlabel('IQ Score', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('PDF: Normal Distribution (IQ Scores)', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# CDF
ax = axes[1]
y_cdf = stats.norm.cdf(x, mu, sigma)

ax.plot(x, y_cdf, 'purple', linewidth=2.5)
ax.fill_between(x, y_cdf, alpha=0.3, color='#9b59b6')
ax.axhline(y=0.5, color='red', linestyle=':', alpha=0.5)
ax.axvline(x=mu, color='red', linestyle='--', linewidth=2, label=f'μ = {mu}, F(μ) = 0.5')
ax.set_xlabel('IQ Score', fontsize=12)
ax.set_ylabel('Cumulative Probability F(x)', fontsize=12)
ax.set_title('CDF: Normal Distribution', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('normal_distribution.png', dpi=100, bbox_inches='tight')
plt.show()

# Compute probabilities
p_above_115 = 1 - stats.norm.cdf(115, mu, sigma)
p_below_85 = stats.norm.cdf(85, mu, sigma)
p_between = stats.norm.cdf(115, mu, sigma) - stats.norm.cdf(85, mu, sigma)

print("Normal Distribution: IQ Scores (μ=100, σ=15)")
print("="*50)
print(f"P(IQ > 115) = 1 - F(115) = {p_above_115:.4f} = {p_above_115:.2%}")
print(f"P(IQ < 85)  = F(85) = {p_below_85:.4f} = {p_below_85:.2%}")
print(f"P(85 ≤ IQ ≤ 115) = F(115) - F(85) = {p_between:.4f} = {p_between:.2%}")
print(f"\nNote: P(IQ = 100) = 0 (point probability is always 0 for continuous vars)")

---

## 8. Exercises

### Exercise 1: Conditional Probability
A manufacturing plant in Italy produces goods with the following quality breakdown:
- 85% of items pass inspection (no defects)
- 10% have minor defects
- 5% have major defects

Among items that are **defective** (minor OR major), what proportion have **major** defects?

Compute P(Major | Defective).

In [ ]:
# YOUR CODE HERE
# Hint: Use the definition P(A|B) = P(A ∩ B) / P(B)
# where A = Major defect, B = Defective

### Exercise 2: Bayes' Theorem in Economics
An economic forecaster claims they can predict recessions 80% of the time (correctly identify recessions when they occur). However, they also false-alarm 10% of the time (predict recession when it doesn't happen).

If the baseline probability of recession in any given year is 5%, what is the probability of a recession given that the forecaster predicts one?

Compute P(Recession | Prediction).

In [ ]:
# YOUR CODE HERE
# Define:
# P(R) = probability of recession
# P(P|R) = probability of prediction given recession (sensitivity)
# P(P|¬R) = probability of prediction given no recession (false alarm rate)
# Compute P(R|P) using Bayes' theorem

### Exercise 3: Independence
An economist collects data on labor market participation. They find:
- P(Female) = 0.45
- P(Works > 35 hours/week) = 0.60
- P(Female AND Works > 35 hours/week) = 0.24

Are "being female" and "working > 35 hours" independent? Justify your answer numerically.

In [ ]:
# YOUR CODE HERE
# Check if P(A ∩ B) = P(A) × P(B)
# If yes, events are independent; if no, they are dependent

### Exercise 4: Continuous Distributions
Suppose household consumption $C$ (in thousands of euros) follows a Normal distribution with mean 30 and standard deviation 8.

Compute:
1. P(C > 40)
2. P(20 ≤ C ≤ 40)
3. What consumption level separates the bottom 25% of households? (25th percentile)

In [ ]:
# YOUR CODE HERE
# Use scipy.stats.norm for CDF and PPF (quantile) functions
# CDF: stats.norm.cdf(value, mean, std)
# Inverse CDF (quantile): stats.norm.ppf(probability, mean, std)

### Exercise 5: Causal Reasoning (Confounder)
A government conducts a study on the effect of vocational training on wages. They observe:
- Workers who take vocational training earn €5,000 more per year on average
- However, vocational training programs are located in wealthy regions
- Workers in wealthy regions earn more even without vocational training

Is this an example of confounding? If so, what is the confounder, and what variable should be included in a regression to isolate the true effect of training?

In [ ]:
# YOUR CODE HERE (conceptual answer)
# Explain:
# 1. What is the confounder?
# 2. Draw a causal DAG showing the relationships
# 3. What variable(s) should be controlled for?

---

## Key Takeaways

1. **Frequentist vs. Bayesian**: Two complementary perspectives on probability. Frequentists focus on long-run frequencies; Bayesians on updating beliefs with data.

2. **Axioms of probability** guarantee that all probability statements satisfy basic logical consistency (non-negativity, normalization, additivity).

3. **Conditional probability** P(A|B) = P(A∩B)/P(B) is fundamental to causal reasoning and shows how information about one event changes our beliefs about another.

4. **Bayes' theorem** P(A|B) = P(B|A)·P(A)/P(B) is the engine of statistical inference: it tells us how to update a prior belief with new evidence.

5. **Independence** P(A∩B) = P(A)·P(B) is a critical assumption in econometrics. Its failure leads to omitted variable bias.

6. **Confounders vs. Mediators**: Confounders create spurious associations and must be controlled; mediators are part of the causal mechanism and should not be controlled if you want the total effect.

7. **Continuous distributions** have densities f(x) where the total area under the curve equals 1. Point probabilities P(X=x) = 0, and density can exceed 1.

8. **The CDF** F(x) = P(X ≤ x) is essential for computing interval probabilities: P(a ≤ X ≤ b) = F(b) - F(a).

---

## Next Module

In the next module, we will study **The Normal Distribution and the Central Limit Theorem**, exploring why the normal distribution appears everywhere in nature and statistics, and how it enables inference even when we don't know the true distribution of the data.

---

**This notebook is part of the open course "Intermediate Statistics for Econometrics" by Andrea Sarcina. Published at cinosar.github.io — Licensed under CC BY 4.0**